# CEIT Corrosion Monitoring Workflow

This notebook shows the CEIT upload flow as clear notebook steps.

**Conceptual steps**

- Resolve the backend ids for the site, location, model definition, and existing analysis.
- Load the CEIT JSON file and turn it into a flat measurement table.
- Match each CEIT sensor code to the SHM sensor records.
- Create or reuse one analysis for the chosen timestamp.
- Build one result series per sensor and metric, then upload or update those rows.
- Read the saved rows back and rebuild the CEIT dataframe.
- Plot the saved results.

**How to run it**

- Run the notebook from top to bottom.
- Set `CREATE_NEW_ANALYSIS = True` to create a new analysis row.
- Set `CREATE_NEW_ANALYSIS = False` to reuse the analysis found for `ANALYSIS_TIMESTAMP`.
- Set `UPLOAD_RESULTS = True` to create or update result rows.
- Set `UPLOAD_RESULTS = False` to skip writes and only inspect the selected analysis.

## Mermaid Color Legend

All workflow diagrams use the same color meaning.

- <span style="color:#0B5CAD;"><strong>Blue</strong></span>: API call.
- <span style="color:#2E7D32;"><strong>Green</strong></span>: data we keep or check.
- <span style="color:#A56A00;"><strong>Yellow</strong></span>: data we build or reshape.
- <span style="color:#C04A2F;"><strong>Red</strong></span>: choice or stop condition.
- <span style="color:#4B5563;"><strong>Grey line</strong></span>: step-to-step flow.

*Read each diagram from top to bottom.*

## Imports

This section loads the APIs, serializers, CEIT helpers, and plotting tools used later in the notebook.

*Outcome:* all notebook steps use the same local SDK code from this workspace.

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display
from owi.metadatabase._utils.utils import load_token_from_env_file
from owi.metadatabase.geometry.io import GeometryAPI
from owi.metadatabase.locations.io import LocationsAPI
from owi.metadatabase.shm import ShmAPI
from rich import print

from owi.metadatabase.results import ResultsAPI
from owi.metadatabase.results.analyses.ceit import (
    CEIT_METRICS,
    CorrosionMonitoring,
    CorrosionMonitoringInput,
    ceit_frame_from_measurements,
    load_ceit_measurements,
)
from owi.metadatabase.results.models import AnalysisDefinition
from owi.metadatabase.results.plotting.ceit import plot_ceit_analyses
from owi.metadatabase.results.serializers import DjangoAnalysisSerializer, DjangoResultSerializer


## Constants

This section sets the file, API root, CEIT metadata, timestamp, and write controls.

**Check before you run**

- `DATA_FILE` points to the CEIT JSON file.
- `PROJECTSITE`, `ASSETLOCATION`, and `MODEL_DEFINITION` point to the right backend objects.
- `ANALYSIS_TIMESTAMP` picks the analysis row to create or reuse.
- `CREATE_NEW_ANALYSIS` decides whether to create a new analysis.
- `UPLOAD_RESULTS` decides whether result rows are written.

*Outcome:* the notebook has one place to review all runtime choices before any API call.*

<html>
    <style>
        table {
            border: none;
            border-color: transparent;
            border-spacing: 2px;
            border-collapse: separate;
            border-radius: 8px;
            font-family: "Latin Modern Mono";
        }
    </style>
</html>


In [3]:

# ** Configuration of the script: identifiers, file paths, API roots, etc.

WORKSPACE_ROOT = Path().resolve().parent
DATA_FILE = WORKSPACE_ROOT / "scripts" / "data" / "MeasFile_24sea.json"
DEFAULT_ENV_FILE = WORKSPACE_ROOT / ".env"
TOKEN_ENV_VAR = "OWI_METADATABASE_API_TOKEN"
DEFAULT_RESULTS_API_ROOT = "https://owimetadatabase-dev.azurewebsites.net/api/v1"

BASE_URL = DEFAULT_RESULTS_API_ROOT
PROJECTSITE = "Willow"
ASSETLOCATION = "CEIT"
MODEL_DEFINITION = "CEIT Willow"
ANALYSIS_NAME = "CeitCorrosionMonitoring-Test-Pietro"
TOKEN = load_token_from_env_file(DEFAULT_ENV_FILE, TOKEN_ENV_VAR)
ANALYSIS_TIMESTAMP = datetime.datetime(2026, 3, 26, 0, 0, 0)
CREATE_NEW_ANALYSIS = False
UPLOAD_RESULTS = True

print("[bold blue]Configuration[/bold blue]")
display(pd.DataFrame([{"data_file": str(DATA_FILE),
                       "api_root": BASE_URL,
                       "projectsite": PROJECTSITE,
                       "assetlocation": ASSETLOCATION,
                       "model_definition": MODEL_DEFINITION,
                       "analysis_name": ANALYSIS_NAME,
                       "analysis_timestamp": ANALYSIS_TIMESTAMP,
                       "create_new_analysis": CREATE_NEW_ANALYSIS,
                       "upload_results": UPLOAD_RESULTS,
                       "token_available": TOKEN is not None}]).T.rename(columns={0: "value"}))


Configuration

,value
data_file,/home/pietro.dantuono@24SEA.local/Projects/SMA...
api_root,https://owimetadatabase-dev.azurewebsites.net/...
projectsite,Willow
assetlocation,CEIT
model_definition,CEIT Willow
analysis_name,CeitCorrosionMonitoring-Test-Pietro
analysis_timestamp,2026-03-26 00:00:00
create_new_analysis,False
upload_results,True
token_available,True


## Resolve CEIT Metadata

Before loading measurements, the notebook resolves the backend ids it needs.

**What is resolved**

- `site_id`
- `location_id`
- `model_definition_id`
- `existing_analysis_id` for the selected `ANALYSIS_TIMESTAMP`

*Outcome:* later cells can work with stable ids instead of repeating name-based lookups.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Notebook configuration"] --> B["Call Locations API"]
    A --> C["Call Geometry API"]
    A --> D["Call Results API"]
    B --> E["site_id and location_id"]
    C --> F["model_definition_id"]
    D --> G["existing_analysis_id for the timestamp"]
    E --> H["Metadata ready"]
    F --> H
    G --> H

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;

    class B,C,D api;
    class E,F,G,H keep;
    class A build;
```

In [4]:

# ** Class instances for API access and CEIT results management

locations_api = LocationsAPI(api_root=BASE_URL, token=TOKEN)
geometry_api = GeometryAPI(api_root=BASE_URL, token=TOKEN)
results_api = ResultsAPI(api_root=BASE_URL, token=TOKEN)
shm_api = ShmAPI(api_root=BASE_URL, token=TOKEN)
analysis = CorrosionMonitoring()
analysis_serializer = DjangoAnalysisSerializer()
result_serializer = DjangoResultSerializer()

# -- site_id
site_detail = locations_api.get_projectsite_detail(projectsite=PROJECTSITE)
site_id = int(site_detail["id"])

# -- location_id
willow_locations = locations_api.get_assetlocations(projectsite=PROJECTSITE)["data"]
ceit_location = willow_locations.loc[
    willow_locations["title"].astype(str).str.casefold() == ASSETLOCATION.casefold()
].copy()
location_id = int(ceit_location.iloc[0]["id"])

# -- model_definition_id
model_definition_id = int(
    geometry_api.get_modeldefinition_id(
        projectsite=PROJECTSITE,
        model_definition=MODEL_DEFINITION,
    )["id"] # type: ignore
)

# -- analysis_id (if existing)
existing_analysis = results_api.get_analysis(
    name=ANALYSIS_NAME,
    model_definition__id=model_definition_id,
    timestamp=ANALYSIS_TIMESTAMP,
    location__id=location_id,
)
existing_analysis_id = (
    None if not existing_analysis["exists"] or existing_analysis["id"] is None else int(existing_analysis["id"])
)

metadata_summary = pd.DataFrame([{"site_id": site_id,
                                  "location_id": location_id,
                                  "model_definition_id": model_definition_id,
                                  "analysis_timestamp": ANALYSIS_TIMESTAMP,
                                  "existing_analysis_id": existing_analysis_id,}]) \
                                 .T.rename(columns={0: "value"})
print("[bold green]Resolved metadata identifiers:[/bold green]")
display(metadata_summary)


Resolved metadata identifiers:

,value
site_id,77
location_id,3443
model_definition_id,18
analysis_timestamp,2026-03-26 00:00:00
existing_analysis_id,69


## Load and Normalize the CEIT Measurements

This stage reads the CEIT JSON file and reshapes it into the flat table used by the rest of the notebook.

**What happens here**

- Load the raw measurements from the JSON file.
- Expand them into one row per sensor, timestamp, and metric.
- List the unique sensor codes found in the file.

*Outcome:* the notebook has one normalized CEIT dataframe ready for matching, upload, retrieval, and plotting.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["CEIT JSON file"] --> B["Load measurements"]
    B --> C["Build normalized dataframe"]
    B --> D["List unique sensor codes"]
    C --> E["Measurement table ready"]
    D --> E

    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;

    class A,B,C,D build;
    class E keep;
```

In [5]:

# ** Load CEIT measurements

measurements = load_ceit_measurements(DATA_FILE)
measurement_frame = ceit_frame_from_measurements(measurements)
unique_sensors = sorted({measurement.sensor_identifier for measurement in measurements})
expected_observation_count = len(measurements) * len(CEIT_METRICS)

measurement_summary = pd.DataFrame([{"raw_measurement_rows": len(measurements),
                                     "unique_sensors": len(unique_sensors),
                                     "normalized_observations": len(measurement_frame),
                                     "expected_observations": expected_observation_count}]) \
                                   .T.rename(columns={0: "value"})
print("[bold green]Loaded CEIT measurements[/bold green]")
display(measurement_summary)
print("[blue]• Sample of normalized measurement frame[/blue]")
display(measurement_frame.head())
print("[blue]• Observation count by sensor and metric[/blue]")
display(measurement_frame.groupby(["sensor_identifier", "metric"]) \
                         .size().rename("points").reset_index() \
                         .sort_values(["sensor_identifier", "metric"]) \
                         .reset_index(drop=True))
del measurement_summary
del expected_observation_count


Loaded CEIT measurements

,value
raw_measurement_rows,24
unique_sensors,3
normalized_observations,120
expected_observations,120


• Sample of normalized measurement frame

,analysis_name,sensor_identifier,metric,timestamp,value,site_id,location_id,related_object_type,related_object_id
0,CeitCorrosionMonitoring,DA8F,temperature,2026-03-12T12:05:12+00:00,-40.84765,None,None,None,None
1,CeitCorrosionMonitoring,DA8F,battery,2026-03-12T12:05:12+00:00,4.13714,None,None,None,None
2,CeitCorrosionMonitoring,DA8F,tof,2026-03-12T12:05:12+00:00,1653.43945,None,None,None,None
3,CeitCorrosionMonitoring,DA8F,amplitude,2026-03-12T12:05:12+00:00,43.00000,None,None,None,None
4,CeitCorrosionMonitoring,DA8F,meas_gain,2026-03-12T12:05:12+00:00,1.39999,None,None,None,None


• Observation count by sensor and metric

,sensor_identifier,metric,points
0,DA87,amplitude,6
1,DA87,battery,6
2,DA87,meas_gain,6
3,DA87,temperature,6
4,DA87,tof,6
5,DA8F,amplitude,12
6,DA8F,battery,12
7,DA8F,meas_gain,12
8,DA8F,temperature,12
9,DA8F,tof,12


## Match CEIT Sensors to SHM Sensors

This section links each CEIT sensor code to the SHM sensor records already stored in the backend.

**What happens here**

- Read the SHM sensor list.
- Match each CEIT sensor code against SHM serial numbers.
- Build a lookup table that shows which backend sensor ids were found.

*Outcome:* each CEIT sensor code can be tied to the backend sensor object used in the result payloads.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Call SHM API"] --> B["Sensor list"]
    C["Unique CEIT sensor codes"] --> D["Match each code"]
    B --> D
    D --> E["Matched sensor ids and serial numbers"]
    E --> F["Sensor lookup table"]

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;

    class A api;
    class B,E,F keep;
    class C,D build;
```

In [6]:

# ** Resolve sensor identifiers to sensor IDs and serial numbers via SHM API

sensor_frame: pd.DataFrame = shm_api.list_sensors()["data"]

def _resolve_sensors(
    sensor_frame: pd.DataFrame, sensor_identifier: str
) -> pd.DataFrame:
    """Resolve sensors based on the given sensor identifier."""
    serial_numbers = sensor_frame["serial_number"].astype(str)
    matches = sensor_frame.loc[
        serial_numbers.str.contains(f"{sensor_identifier}-", na=False)
    ].copy().drop_duplicates(subset=["id"]).sort_values(["serial_number", "id"])  # ty:ignore[no-matching-overload]
    return matches.reset_index(drop=True)

resolution_rows: list[dict[str, int | str]] = []  # ty:ignore

for sensor_identifier in unique_sensors:
    candidates = _resolve_sensors(sensor_frame, sensor_identifier)
    matched_sensor_ids = [] if candidates.empty else [
        int(value) for value in candidates["id"].tolist()
    ]
    matched_serial_numbers = [] if candidates.empty else [
        str(value) for value in candidates["serial_number"].tolist()
    ]

    resolution_rows.append({"sensor_identifier": sensor_identifier,
                            "matched_sensor_count": len(matched_sensor_ids),
                            "matched_sensor_ids": ", ".join(str(value) for value in matched_sensor_ids),
                            "matched_serial_numbers": ", ".join(matched_serial_numbers)})

sensor_identifier_to_id_map = pd.DataFrame(resolution_rows).sort_values(
    "sensor_identifier"
).reset_index(drop=True)
display(sensor_identifier_to_id_map)


,sensor_identifier,matched_sensor_count,matched_sensor_ids,matched_serial_numbers
0,DA87,1,92,DA87-06
1,DA8F,1,90,DA8F-01
2,DA9D,1,91,DA9D-05


## Build and Upload the Shared Analysis

This section builds one generic analysis payload, enriches each CEIT row with its related object, serializes the derived result series, and persists them through `ResultsAPI`.

**Execution logic**

- Build the shared `AnalysisDefinition` with `ANALYSIS_TIMESTAMP` so the backend lookup is unique.
- Convert the resolved CEIT rows into `CorrosionMonitoring` `ResultSeries` objects.
- Serialize the analysis and result payloads with the generic Django serializers.
- When `CREATE_NEW_ANALYSIS` is `True`, create a new analysis row and fail fast if the timestamped analysis already exists.
- When `CREATE_NEW_ANALYSIS` is `False`, reuse the existing timestamped analysis row.
- When `UPLOAD_RESULTS` is `True`, call `results_api.create_or_update_results_bulk(upload_payloads)` so missing stable sensor-metric rows are created and existing rows are patched.
- When `UPLOAD_RESULTS` is `False`, skip result POST/PATCH operations and continue with retrieval against the selected analysis id.

*Outcome:* one shared analysis stores all CEIT sensors while analysis creation and row upload stay independently controllable.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Prepared analysis and result payloads"] --> B{"Create new analysis?"}
    B -- yes --> C["Create the analysis row"]
    B -- no --> D["Reuse the existing analysis id"]
    C --> E["Selected analysis id"]
    D --> E
    E --> F{"Upload result rows?"}
    F -- yes --> G["Bulk create missing rows or patch existing rows"]
    F -- no --> H["Skip writes"]
    G --> I["Upload summary"]
    H --> I

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;
    classDef decision fill:#F7D9D1,stroke:#C04A2F,color:#5A1F14;

    class C,G api;
    class D,E,I keep;
    class A,H build;
    class B,F decision;
```

In [7]:

# ** Create analysis (if configured to do so and if not already existing)

analysis_definition = AnalysisDefinition(
    name=ANALYSIS_NAME,
    model_definition_id=model_definition_id,
    location_id=location_id,
    source_type="json",
    source=str(DATA_FILE),
    timestamp=ANALYSIS_TIMESTAMP,
    description="Shared CEIT corrosion monitoring upload.",
    additional_data={"input_file": DATA_FILE.name},
)
analysis_payload = analysis_serializer.to_payload(analysis_definition)

if CREATE_NEW_ANALYSIS and existing_analysis_id is not None:
    raise ValueError(
        "An analysis already exists for the configured name, timestamp, model definition, and location. "
        "Set CREATE_NEW_ANALYSIS to False or choose a different ANALYSIS_TIMESTAMP."
    )

if CREATE_NEW_ANALYSIS:
    created_analysis = results_api.create_analysis(analysis_payload)
    analysis_id = int(created_analysis["id"])
    analysis_created = True
else:
    if existing_analysis_id is None:
        raise ValueError("No existing analysis matched the configured name, timestamp, "
                         "model definition, and location.\nSet CREATE_NEW_ANALYSIS to True "
                         "or adjust ANALYSIS_TIMESTAMP.")
    analysis_id = existing_analysis_id
    analysis_created = False


In [8]:

# ** Enrich measurements with related SHM sensor objects and prepare results series for upload

resolution_lookup = sensor_identifier_to_id_map.set_index("sensor_identifier").to_dict(orient="index")
related_objects_by_sensor: dict[str, dict[str, int | str]] = {}  # ty:ignore
enriched_measurements = []

for measurement in measurements:
    candidates = _resolve_sensors(sensor_frame, measurement.sensor_identifier)
    if len(candidates) != 1:
        raise ValueError(f"Expected exactly one SHM sensor for {measurement.sensor_identifier}, "
                         f"found {len(candidates)}.")
    sensor_row = candidates.iloc[0]
    related_object = {"type": "shm.sensor", "id": int(sensor_row["id"])}
    related_objects_by_sensor[measurement.sensor_identifier] = related_object
    enriched_measurements.append(measurement.model_copy(update={"site_id": site_id,
                                                                "location_id": location_id,
                                                                "related_object": related_object}))

analysis_input = CorrosionMonitoringInput(rows=enriched_measurements)
result_series = analysis.to_results(analysis_input)
for series in result_series:
    sensor_identifier = str(series.data_additional["sensor_identifier"])
    resolution_row = resolution_lookup[sensor_identifier]
    series.data_additional.update({"input_file": DATA_FILE.name,
                                   "sensor_serial_numbers": resolution_row["matched_serial_numbers"]})


In [9]:

# ** Upload results to the API (if configured to do so)

results_payloads = [result_serializer.to_payload(series, analysis_id=analysis_id)
                   for series in result_series]
upload_result = {"summary": [], "data": pd.DataFrame(), "exists": False}
if UPLOAD_RESULTS:
    upload_result = results_api.create_or_update_results_bulk(results_payloads)

# --- Sumary

summary_frame = pd.DataFrame(upload_result["summary"],
                             columns=["analysis", "short_description", "result_id", "action"])
payload_summary = pd.DataFrame([{"analysis": int(payload["analysis"]),
                                 "short_description": str(payload["short_description"]),
                                 "sensor_identifier": payload["additional_data"]["sensor_identifier"],
                                 "metric": payload["additional_data"]["metric"]}
                                for payload in results_payloads])
upload_summary = payload_summary.merge(summary_frame,
                                       on=["analysis", "short_description"],
                                       how="left")
if not upload_summary.empty:
    upload_summary = upload_summary.rename(columns={"short_description": "series_key"})
    upload_summary = upload_summary.sort_values(["sensor_identifier", "metric", "series_key"]) \
                                   .reset_index(drop=True)

print("[bold green]Preview of analysis and results payloads[/bold green]")
display(pd.DataFrame([analysis_payload]))
print("[blue]• Sample of results payloads for the first sensor[/blue]")
display(pd.DataFrame(results_payloads[:5]))
print("[blue]• Upload summary[/blue]")
display(upload_summary)
print("[blue]• Summary of uploaded series by sensor and metric[/blue]")
display(pd.DataFrame([{"analysis_id": analysis_id,
                       "analysis_created": analysis_created,
                       "analysis_timestamp": ANALYSIS_TIMESTAMP,
                       "upload_results": UPLOAD_RESULTS,
                       "series_uploaded": len(upload_summary),
                       "linked_related_objects": len(related_objects_by_sensor),
                       "expected_series": len(unique_sensors) * len(CEIT_METRICS)}]) \
                     .T.rename(columns={0: "value"}))


Uploading existing result rows:   0%|          | 0/15 [00:00<?, ?row/s]

Preview of analysis and results payloads

,name,model_definition_id,location_id,source_type,source,description,timestamp,additional_data
0,CeitCorrosionMonitoring-Test-Pietro,18,3443,json,/home/pietro.dantuono@24SEA.local/Projects/SMA...,Shared CEIT corrosion monitoring upload.,2026-03-26,{'input_file': 'MeasFile_24sea.json'}


• Sample of results payloads for the first sensor

,analysis,site,location,related_object,name_col1,name_col2,units_col1,units_col2,value_col1,value_col2,short_description,description,additional_data
0,69,77,3443,"{'type': 'shm.sensor', 'id': 92}",timestamp,temperature,s,degC,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[-40.84765, -40.84765, -40.84765, -40.84765, -...",DA87:temperature,CEIT temperature time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."
1,69,77,3443,"{'type': 'shm.sensor', 'id': 92}",timestamp,battery,s,V,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[4.13714, 4.13714, 4.13714, 4.13714, 4.13714, ...",DA87:battery,CEIT battery time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."
2,69,77,3443,"{'type': 'shm.sensor', 'id': 92}",timestamp,tof,s,us,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[1653.35546, 1653.35546, 1653.35546, 1653.3554...",DA87:tof,CEIT tof time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."
3,69,77,3443,"{'type': 'shm.sensor', 'id': 92}",timestamp,amplitude,s,count,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[43.0, 43.0, 43.0, 43.0, 43.0, 43.0]",DA87:amplitude,CEIT amplitude time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."
4,69,77,3443,"{'type': 'shm.sensor', 'id': 92}",timestamp,meas_gain,s,gain,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[1.39999, 1.39999, 1.39999, 1.39999, 1.39999, ...",DA87:meas_gain,CEIT meas_gain time series for sensor DA87.,"{'analysis_kind': 'time_series', 'result_scope..."


• Upload summary

,analysis,series_key,sensor_identifier,metric,result_id,action
0,69,DA87:amplitude,DA87,amplitude,5729,updated
1,69,DA87:battery,DA87,battery,5727,updated
2,69,DA87:meas_gain,DA87,meas_gain,5730,updated
3,69,DA87:temperature,DA87,temperature,5726,updated
4,69,DA87:tof,DA87,tof,5728,updated
5,69,DA8F:amplitude,DA8F,amplitude,5734,updated
6,69,DA8F:battery,DA8F,battery,5732,updated
7,69,DA8F:meas_gain,DA8F,meas_gain,5735,updated
8,69,DA8F:temperature,DA8F,temperature,5731,updated
9,69,DA8F:tof,DA8F,tof,5733,updated


• Summary of uploaded series by sensor and metric

,value
analysis_id,69
analysis_created,False
analysis_timestamp,2026-03-26 00:00:00
upload_results,True
series_uploaded,15
linked_related_objects,3
expected_series,15


## Retrieve the Persisted Results

This stage reads the saved rows back from the API and rebuilds the CEIT dataframe from them.

**What happens here**

- Read the raw result rows for the selected analysis.
- Convert those rows back into result series objects.
- Rebuild the CEIT dataframe from the saved series.

*Outcome:* the notebook checks the saved backend data, not just the original input file.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Selected analysis id"] --> B["Read saved result rows"]
    B --> C["Raw backend table"]
    C --> D["Convert rows back to result series"]
    D --> E["Rebuild the CEIT dataframe"]
    E --> F["Retrieved dataframe ready"]

    classDef api fill:#CFE0F5,stroke:#0B5CAD,color:#062B5B;
    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;

    class B api;
    class C,F keep;
    class A,D,E build;
```

In [10]:

# ** Retrieve results for a specific analysis from the API and convert them back to ResultSeries and DataFrame

raw_results_frame = results_api.list_results(analysis=analysis_id)["data"]
print("[bold green]Raw results as retrieved from the API[/bold green]")
display(raw_results_frame.head())
retrieved_series = [result_serializer.from_mapping(record)
                    for record in raw_results_frame.to_dict(orient="records")]
print("[blue]• Example raw results converted to ResultSeries[/blue]")
display(retrieved_series[:2])
retrieved_frame = analysis.from_results(retrieved_series)
print("[blue]• ResultSeries converted to DataFrame[/blue]")
display(retrieved_frame.head())


Raw results as retrieved from the API

,id,name_col1,name_col2,name_col3,units_col1,units_col2,units_col3,value_col1,value_col2,value_col3,analysis,site,location,short_description,description,related_object,additional_data,slug,visibility,visibility_groups
0,5729,timestamp,amplitude,None,s,count,None,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[43.0, 43.0, 43.0, 43.0, 43.0, 43.0]",None,69,77,3443,DA87:amplitude,CEIT amplitude time series for sensor DA87.,"{'type': 'shm.sensor', 'id': 92}","{'metric': 'amplitude', 'input_file': 'MeasFil...",willow_ceit_ceitcorrosionmonitoring-test-pietr...,usergroup,[31]
1,5727,timestamp,battery,None,s,V,None,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[4.13714, 4.13714, 4.13714, 4.13714, 4.13714, ...",None,69,77,3443,DA87:battery,CEIT battery time series for sensor DA87.,"{'type': 'shm.sensor', 'id': 92}","{'metric': 'battery', 'input_file': 'MeasFile_...",willow_ceit_ceitcorrosionmonitoring-test-pietr...,usergroup,[31]
2,5730,timestamp,meas_gain,None,s,gain,None,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[1.39999, 1.39999, 1.39999, 1.39999, 1.39999, ...",None,69,77,3443,DA87:meas_gain,CEIT meas_gain time series for sensor DA87.,"{'type': 'shm.sensor', 'id': 92}","{'metric': 'meas_gain', 'input_file': 'MeasFil...",willow_ceit_ceitcorrosionmonitoring-test-pietr...,usergroup,[31]
3,5726,timestamp,temperature,None,s,degC,None,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[-40.84765, -40.84765, -40.84765, -40.84765, -...",None,69,77,3443,DA87:temperature,CEIT temperature time series for sensor DA87.,"{'type': 'shm.sensor', 'id': 92}","{'metric': 'temperature', 'input_file': 'MeasF...",willow_ceit_ceitcorrosionmonitoring-test-pietr...,usergroup,[31]
4,5728,timestamp,tof,None,s,us,None,"[1773317292.0, 1773317292.0, 1773403692.0, 177...","[1653.35546, 1653.35546, 1653.35546, 1653.3554...",None,69,77,3443,DA87:tof,CEIT tof time series for sensor DA87.,"{'type': 'shm.sensor', 'id': 92}","{'metric': 'tof', 'input_file': 'MeasFile_24se...",willow_ceit_ceitcorrosionmonitoring-test-pietr...,usergroup,[31]


• Example raw results converted to ResultSeries

[ResultSeries(analysis_name='unknown', analysis_kind=<AnalysisKind.TIME_SERIES: 'time_series'>, result_scope=<ResultScope.LOCATION: 'location'>, short_description='DA87:amplitude', description='CEIT amplitude time series for sensor DA87.', site_id=77, location_id=3443, related_object=RelatedObject(type='shm.sensor', id=92), data_additional={'metric': 'amplitude', 'input_file': 'MeasFile_24sea.json', 'series_key': 'DA87:amplitude', 'result_scope': 'location', 'source_field': 'Amplitude', 'analysis_kind': 'time_series', 'related_object_id': 92, 'sensor_identifier': 'DA87', 'related_object_type': 'shm.sensor', 'sensor_serial_numbers': 'DA87-06'}, vectors=[ResultVector(name='timestamp', unit='s', values=[1773317292.0, 1773317292.0, 1773403692.0, 1773403692.0, 1773576492.0, 1773576492.0]), ResultVector(name='amplitude', unit='count', values=[43.0, 43.0, 43.0, 43.0, 43.0, 43.0])]),
 ResultSeries(analysis_name='unknown', analysis_kind=<AnalysisKind.TIME_SERIES: 'time_series'>, result_scope=<R

• ResultSeries converted to DataFrame

,analysis_name,sensor_identifier,metric,timestamp,value,site_id,location_id,related_object_type,related_object_id
0,unknown,DA87,amplitude,2026-03-12T12:08:12+00:00,43.0,77,3443,shm.sensor,92
1,unknown,DA87,amplitude,2026-03-12T12:08:12+00:00,43.0,77,3443,shm.sensor,92
2,unknown,DA87,amplitude,2026-03-13T12:08:12+00:00,43.0,77,3443,shm.sensor,92
3,unknown,DA87,amplitude,2026-03-13T12:08:12+00:00,43.0,77,3443,shm.sensor,92
4,unknown,DA87,amplitude,2026-03-15T12:08:12+00:00,43.0,77,3443,shm.sensor,92


## Plot the Retrieved Results

The final stage renders the CEIT results from the retrieved backend rows, not from the raw input file.

**What to inspect in the plot**

- The dropdown should expose one entry per sensor identifier.
- Each sensor chart should show the five CEIT metrics as separate time-series lines.
- The plotted data should match the timestamps and values shown in the retrieved dataframe preview.

*Outcome:* the same persisted analysis can be inspected interactively through the CEIT plotting layer exposed by the SDK.*

```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'lineColor': '#4B5563',
  'clusterBkg': '#F8FAFC',
  'clusterBorder': '#CBD5E1'
}}}%%
graph TD
    A["Retrieved dataframe"] --> B["Build the CEIT plot"]
    B --> C["Interactive chart"]
    C --> D["Display the plot"]

    classDef keep fill:#DCEFD8,stroke:#2E7D32,color:#103816;
    classDef build fill:#F3E3BF,stroke:#A56A00,color:#4A3200;

    class A,C,D keep;
    class B build;
```

In [11]:
shared_plot = plot_ceit_analyses(retrieved_frame)
display(shared_plot.notebook)


In [12]:
final_summary = pd.DataFrame([{"analysis_id": analysis_id,
                               "analysis_created": analysis_created,
                               "analysis_timestamp": ANALYSIS_TIMESTAMP,
                               "create_new_analysis": CREATE_NEW_ANALYSIS,
                               "upload_results": UPLOAD_RESULTS,
                               "persisted_series": 0 if raw_results_frame.empty \
                                                     else raw_results_frame["short_description"].nunique(),
                               "input_rows": len(measurements),
                               "normalized_input_observations": len(measurement_frame),
                               "retrieved_observations": len(retrieved_frame),
                               "plot_available": shared_plot.notebook is not None}]) \
                             .T.rename(columns={0: "value"})
display(final_summary)
del final_summary


,value
analysis_id,69
analysis_created,False
analysis_timestamp,2026-03-26 00:00:00
create_new_analysis,False
upload_results,True
persisted_series,15
input_rows,24
normalized_input_observations,120
retrieved_observations,120
plot_available,True
